In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [28]:
hourly_df = pd.read_csv("hourly_taxi_demand.csv")

hourly_df["tpep_pickup_datetime"] = pd.to_datetime(hourly_df["tpep_pickup_datetime"])

hourly_df = hourly_df.set_index("tpep_pickup_datetime")

In [29]:
forecast_df = pd.read_csv("demand_forecast.csv")

forecast_df["tpep_pickup_datetime"] = pd.to_datetime(
    forecast_df["tpep_pickup_datetime"]
)

forecast_df = forecast_df.set_index("tpep_pickup_datetime")

forecast_demand = forecast_df["forecast_trips"]

In [30]:
hourly_df["hour_of_week"] = (
    hourly_df.index.dayofweek * 24 + hourly_df.index.hour
)

In [31]:
train = hourly_df[:-168]
test = hourly_df[-168:]

In [32]:
seasonal_baseline = train.groupby("hour_of_week")["trips"].mean()

baseline_forecast = test.index.map(
    lambda x: seasonal_baseline[x.dayofweek * 24 + x.hour]
)

In [33]:
surge_multiplier = 1 + 0.6 * (
    (forecast_demand - baseline_forecast) / baseline_forecast
)

surge_multiplier = surge_multiplier.clip(0.8, 1.5)

In [34]:
avg_fare = train["avg_fare"].mean()

In [35]:
static_revenue = forecast_demand * avg_fare

In [36]:
dynamic_revenue_no_elasticity = (
    forecast_demand * avg_fare * surge_multiplier
)

In [37]:
forecast_demand = forecast_demand.clip(lower=0)

elasticity = -0.1

adjusted_demand = forecast_demand * (surge_multiplier ** elasticity)

dynamic_revenue_elasticity = (
    adjusted_demand * avg_fare * surge_multiplier
)

In [38]:
static_total = static_revenue.sum()

dynamic_no_elasticity_total = dynamic_revenue_no_elasticity.sum()

dynamic_elasticity_total = dynamic_revenue_elasticity.sum()

print("Static Revenue:", static_total)
print("Dynamic Revenue (No Elasticity):", dynamic_no_elasticity_total)
print("Dynamic Revenue (Elasticity):", dynamic_elasticity_total)

Static Revenue: 26374719.05280175
Dynamic Revenue (No Elasticity): 27061507.44502665
Dynamic Revenue (Elasticity): 26967882.218610898


In [39]:
lift_no_elasticity = dynamic_no_elasticity_total / static_total - 1

lift_elasticity = dynamic_elasticity_total / static_total - 1

print("Revenue Lift (No Elasticity):", lift_no_elasticity)
print("Revenue Lift (Elasticity):", lift_elasticity)

Revenue Lift (No Elasticity): 0.026039647696339818
Revenue Lift (Elasticity): 0.022489838266017026


In [40]:
results = pd.DataFrame({
    "Static Revenue":[static_total],
    "Dynamic Revenue No Elasticity":[dynamic_no_elasticity_total],
    "Dynamic Revenue Elasticity":[dynamic_elasticity_total],
    "Lift No Elasticity":[lift_no_elasticity],
    "Lift Elasticity":[lift_elasticity]
})

results.to_csv("revenue_results.csv", index=False)